# Retention Model Interpretation and Threshold Analysis

This notebook interprets the selected employee-attrition model and evaluates alternative classification thresholds.

The analysis focuses on:

- Feature importance
- Feature direction for Logistic Regression
- Original feature-group importance
- Precision-recall trade-offs
- False-positive and false-negative trade-offs
- Recommended classification threshold
- Low-, medium-, and high-risk employee segments

Model interpretation describes associations learned from the synthetic data and should not be interpreted as evidence of causation.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


feature_importance = pd.read_csv(
    PROCESSED_DIR
    / "retention_feature_importance.csv"
)


feature_group_importance = pd.read_csv(
    PROCESSED_DIR
    / "retention_feature_group_importance.csv"
)


threshold_analysis = pd.read_csv(
    PROCESSED_DIR
    / "retention_threshold_analysis.csv"
)


risk_segments = pd.read_csv(
    PROCESSED_DIR
    / "retention_risk_segments.csv"
)


print(
    "Feature importance rows:",
    len(feature_importance),
)

print(
    "Feature groups:",
    len(feature_group_importance),
)

print(
    "Thresholds evaluated:",
    len(threshold_analysis),
)

print(
    "Risk segment rows:",
    len(risk_segments),
)

Feature importance rows: 70
Feature groups: 35
Thresholds evaluated: 17
Risk segment rows: 1478


## 1. Detailed feature importance

In [2]:
feature_importance[
    [
        "transformed_feature",
        "importance",
        "direction",
        "source_feature",
    ]
].head(20)

,transformed_feature,importance,direction,source_feature
0,categorical__hire_job_family_Manufacturing,1.897311,Lower attrition risk,hire_job_family
1,numerical__initial_base_salary,1.767611,Lower attrition risk,initial_base_salary
2,categorical__hire_job_level_3,1.367819,Higher attrition risk,hire_job_level
3,categorical__hire_job_family_Customer Support,1.155447,Lower attrition risk,hire_job_family
4,categorical__hire_department_name_Customer Sup...,1.155447,Lower attrition risk,hire_department_name
5,numerical__base_salary,1.109229,Higher attrition risk,base_salary
6,categorical__hire_job_family_Engineering,1.045394,Higher attrition risk,hire_job_family
7,numerical__equity_value,1.027509,Lower attrition risk,equity_value
8,categorical__employment_type_Hourly,0.985982,Higher attrition risk,employment_type
9,categorical__employment_type_Salaried,0.973474,Lower attrition risk,employment_type


## 2. Original feature-group importance

In [3]:
feature_group_importance.head(20)

,source_feature,total_importance,transformed_feature_count
0,hire_job_family,6.656404,10
1,hire_department_name,2.732181,8
2,hire_job_level,2.723130,4
3,employment_type,1.959455,2
4,initial_base_salary,1.767611,1
5,application_source,1.282041,8
6,base_salary,1.109229,1
7,equity_value,1.027509,1
8,hire_region,0.625855,4
9,education_level,0.557557,5


## 3. Classification threshold analysis

In [4]:
threshold_analysis[
    [
        "threshold",
        "precision",
        "recall",
        "f1",
        "false_positive",
        "false_negative",
        "true_positive",
    ]
]

,threshold,precision,recall,f1,false_positive,false_negative,true_positive
0,0.10,0.0910,1.000,0.1669,1248,0,125
1,0.15,0.0927,1.000,0.1697,1223,0,125
2,0.20,0.0961,0.992,0.1753,1166,1,124
3,0.25,0.1004,0.984,0.1822,1102,2,123
4,0.30,0.1049,0.960,0.1891,1024,5,120
5,0.35,0.1099,0.912,0.1962,923,11,114
6,0.40,0.1196,0.880,0.2105,810,15,110
7,0.45,0.1307,0.808,0.2249,672,24,101
8,0.50,0.1376,0.712,0.2306,558,36,89
9,0.55,0.1437,0.568,0.2294,423,54,71


In [5]:
eligible_thresholds = (
    threshold_analysis[
        threshold_analysis[
            "recall"
        ]
        >= 0.60
    ]
)


if not eligible_thresholds.empty:

    recommended_threshold_row = (
        eligible_thresholds
        .sort_values(
            [
                "precision",
                "f1",
                "threshold",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .iloc[0]
    )

else:

    recommended_threshold_row = (
        threshold_analysis
        .sort_values(
            "f1",
            ascending=False,
        )
        .iloc[0]
    )


recommended_threshold_row

threshold                      0.5000
accuracy                       0.5981
precision                      0.1376
recall                         0.7120
f1                             0.2306
predicted_attrition_count    647.0000
true_negative                795.0000
false_positive               558.0000
false_negative                36.0000
true_positive                 89.0000
Name: 8, dtype: float64

In [6]:
default_threshold_row = (
    threshold_analysis[
        threshold_analysis[
            "threshold"
        ]
        == 0.50
    ]
    .iloc[0]
)


threshold_comparison = pd.DataFrame(
    [
        {
            "threshold_type": (
                "Default"
            ),
            **default_threshold_row.to_dict(),
        },

        {
            "threshold_type": (
                "Recommended"
            ),
            **recommended_threshold_row.to_dict(),
        },
    ]
)


threshold_comparison

,threshold_type,threshold,accuracy,precision,recall,f1,predicted_attrition_count,true_negative,false_positive,false_negative,true_positive
0,Default,0.5,0.5981,0.1376,0.712,0.2306,647.0,795.0,558.0,36.0,89.0
1,Recommended,0.5,0.5981,0.1376,0.712,0.2306,647.0,795.0,558.0,36.0,89.0


## 4. Employee risk segments

In [7]:
risk_segment_summary = (
    risk_segments
    .groupby(
        "risk_segment"
    )
    .agg(
        employee_count=(
            "employee_id",
            "count",
        ),

        average_predicted_probability=(
            "attrition_probability",
            "mean",
        ),

        actual_attrition_rate=(
            "actual_attrition",
            "mean",
        ),
    )
)


risk_segment_summary

,employee_count,average_predicted_probability,actual_attrition_rate
risk_segment,,,
High,148,0.738725,0.182432
Low,1034,0.350687,0.057060
Medium,296,0.622756,0.131757


In [8]:
risk_segment_summary[
    "actual_attrition_rate_percent"
] = (
    100
    * risk_segment_summary[
        "actual_attrition_rate"
    ]
)


risk_segment_summary

,employee_count,average_predicted_probability,actual_attrition_rate,actual_attrition_rate_percent
risk_segment,,,,
High,148,0.738725,0.182432,18.243243
Low,1034,0.350687,0.057060,5.705996
Medium,296,0.622756,0.131757,13.175676


## 5. Interpretation validation

In [9]:
risk_rates = (
    risk_segment_summary[
        "actual_attrition_rate"
    ]
)


interpretation_checks = pd.Series(
    {
        "feature importance is not empty": (
            len(
                feature_importance
            )
            > 0
        ),

        "feature groups are not empty": (
            len(
                feature_group_importance
            )
            > 0
        ),

        "threshold analysis is not empty": (
            len(
                threshold_analysis
            )
            > 0
        ),

        "threshold values are valid": (
            threshold_analysis[
                "threshold"
            ]
            .between(
                0,
                1,
            )
            .all()
        ),

        "precision values are valid": (
            threshold_analysis[
                "precision"
            ]
            .between(
                0,
                1,
            )
            .all()
        ),

        "recall values are valid": (
            threshold_analysis[
                "recall"
            ]
            .between(
                0,
                1,
            )
            .all()
        ),

        "three risk segments appear": (
            set(
                risk_segments[
                    "risk_segment"
                ]
            )
            == {
                "Low",
                "Medium",
                "High",
            }
        ),
    },
    name="passed",
)


interpretation_checks

feature importance is not empty    True
feature groups are not empty       True
threshold analysis is not empty    True
threshold values are valid         True
precision values are valid         True
recall values are valid            True
three risk segments appear         True
Name: passed, dtype: bool

In [10]:
if interpretation_checks.all():

    print(
        "All retention model interpretation "
        "checks passed."
    )

else:

    print(
        "One or more interpretation "
        "checks failed."
    )

All retention model interpretation checks passed.


## 6. Conclusions

The selected retention model has been evaluated beyond the default 0.50 classification threshold.

### Model interpretation

Feature importance identifies the variables and categories that most strongly influence model predictions.

For Logistic Regression, positive coefficients are associated with higher predicted attrition risk, while negative coefficients are associated with lower predicted attrition risk.

These relationships describe patterns learned from synthetic data and do not establish causation.

### Threshold analysis

Changing the classification threshold changes the balance between:

- Precision
- Recall
- False positives
- False negatives

The project uses a practical threshold-selection rule that maintains at least 60% recall and then selects the threshold with the highest precision.

This rule is a project assumption and could be changed based on business priorities.

### Risk segmentation

Employees in the test population are separated into:

- Low risk: bottom 70% of predicted scores
- Medium risk: next 20%
- High risk: top 10%

Risk bands provide relative prioritization and should not be interpreted as perfectly calibrated probabilities.

### Next step

The next stage will prepare model outputs and workforce metrics for a decision-support dashboard.